## 2.2.1 读取数据集

In [12]:
import os

#os.mkdir('a/b/c/data')（单个）​它只能创建最后一级，前提是前面的 a/b/c 必须已经存在。
#os.makedirs('a/b/c/data')（多个）​它可以一次性把缺的中间层级全部自动创建出来。

#os.path.join('..', 'data') → 把两段拼成一个路径
#exist_ok=True → 如果目录已经存在，不报错
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
#with ... as f: → 上下文管理器，代码块结束时自动关闭文件
# 'w' 会清空原有内容（如果文件已存在）再写；文件不存在则新建
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')
    f.write('NA,Pave,127500\n')
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


## 2.2.2处理缺失值

典型的方法：
> 1.插值法：用一个替代值弥补缺失值：
> 2.删除法：直接忽略缺失值。

In [13]:
#NaN → 缺失值 
#插值法
#前者为data的前两列，而后者为data的最后一列
# inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs, outputs = data.loc[:,'NumRooms':'Alley'], data.loc[:,'Price']

#fillna 是 pandas 里用来填充缺失值（NaN）​的函数。
# 参数是填入什么，还可以指定列，比如 {'A': 0, 'B': 100}
#会返回新对象
#numeric_only=True → 只对数值列进行插值
inputs = inputs.fillna(inputs.mean(numeric_only=True))
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


iloc 的切片是"左闭右开"​（和 numpy / 列表一样），按"位置"索引，   
loc 的切片是两端都包含，按标签（行列名）索引。

“巷子类型”(“Alley”)列只接受两种类型的类别值“Pave”和“NaN”，pandas可以自动将此列转换为两列“Alley_Pave”和“Alley_nan”。

In [14]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


## 2.2.3转为张量形式 to_numpy()
这样就可以用张量函数来进一步操作

In [15]:
import torch
x =torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
print(x)
print(y)

tensor([[3., 1., 0.],
        [2., 0., 1.],
        [4., 0., 1.],
        [3., 0., 1.]], dtype=torch.float64)
tensor([127500., 106000., 178100., 140000.], dtype=torch.float64)
